In [1]:
import os

# LLM keys for the agent + tools
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"
os.environ["LLM_API_KEY"] = os.environ["OPENAI_API_KEY"]
os.environ["LLM_MODEL"] = "gpt-4o-mini"

from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

from cognee_integration_langgraph import add_tool, search_tool


2026-03-13T07:46:43.049642 [warning  ] From version 0.5.0 onwards, Cognee will run with multi-user access control mode set to on by default. Data isolation between different users and datasets will be enforced and data created before multi-user access control mode was turned on won't be accessible by default. To disable multi-user access control mode and regain access to old data set the environment variable ENABLE_BACKEND_ACCESS_CONTROL to false before starting Cognee. For more information, please refer to the Cognee documentation. [cognee.shared.logging_utils]

2026-03-13T07:46:43.050430 [info     ] Log file created at: /Users/handekafkas/Documents/local-code/cognee-integrations/integrations/langgraph/.venv/lib/python3.10/site-packages/logs/2026-03-13_08-46-42.log [cognee.shared.logging_utils] log_file=/Users/handekafkas/Documents/local-code/cognee-integrations/integrations/langgraph/.venv/lib/python3.10/site-packages/logs/2026-03-13_08-46-42.log

2026-03-13T07:46:43.050929 [info   

In [2]:
agent = create_react_agent(
    "openai:gpt-4o-mini",
    tools=[add_tool, search_tool],
)
agent.step_timeout = None

/var/folders/85/g_scmr8n4w1bk9_6vq_klj5r0000gn/T/ipykernel_66290/3363876105.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [3]:
response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="""
We have signed a contract with the following company: "Acme Rocket Skates Ltd".
Company is in the cartoon physics industry.
Start date is Oct 2024 and end date is Oct 2029.
Contract value is £4.2M.
"""
            ),
            HumanMessage(
                content="""
We have signed a contract with the following company: "Globex Analytics".
Industry: enterprise data platforms.
Start date is May 2024 and end date is May 2027.
Contract value is £2.3M.
"""
            ),
        ]
    }
)
print(response["messages"][-1].content)


Adding data to cognee: Company: Acme Rocket Skates Ltd, Industry: cartoon physics, Start date: Oct 2024, End date: Oct 2029, Contract value: £4.2M.

Adding data to cognee: Company: Globex Analytics, Industry: enterprise data platforms, Start date: May 2024, End date: May 2027, Contract value: £2.3M.

2026-03-13T07:47:06.249433 [info     ] Testing connection to LLM endpoint... [cognee.shared.logging_utils]


User 016ceaf9-add3-4fda-a9bb-9a23a6bb570e has registered.



2026-03-13T07:47:13.174093 [info     ] Testing connection to Embedding endpoint... [cognee.shared.logging_utils]

2026-03-13T07:47:15.607224 [info     ] Pipeline run started: `d163a974-164b-5cc1-8835-b1726d914fd3` [run_tasks_with_telemetry()]

2026-03-13T07:47:15.879708 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-03-13T07:47:16.071074 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-03-13T07:47:16.288885 [info     ] Registered loader: pypdf_loader [cognee.infrastructure.loaders.LoaderEngine]

2026-03-13T07:47:16.289338 [info     ] Registered loader: text_loader [cognee.infrastructure.loaders.LoaderEngine]

2026-03-13T07:47:16.289675 [info     ] Registered loader: image_loader [cognee.infrastructure.loaders.LoaderEngine]

2026-03-13T07:47:16.290069 [info     ] Registered loader: audio_loader [cognee.infrastructure.loaders.LoaderEngine]

2026-03-13T07:47:16.290406 [info     ] Registered loader: csv_loader  [cognee.infrast

The contracts for both "Acme Rocket Skates Ltd" and "Globex Analytics" have been successfully recorded. If you need any further assistance or have more information to add, feel free to let me know!


In [6]:
response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="""
Summarize our contracts, including company names, terms, and values.
Use your search functionality to find the details you just stored.
"""
            )
        ]
    }
)
print(response["messages"][-1].content)


Searching cognee for: contracts, company names, terms, and values with session_id: None, node_set: None, query_type: None

2026-03-13T07:52:45.697515 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.33s [cognee.shared.logging_utils]

2026-03-13T07:52:45.698375 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-03-13T07:52:45.707126 [info     ] ID-filtered retrieval: 20 nodes and 32 edges in 0.01s [cognee.shared.logging_utils]

2026-03-13T07:52:45.708562 [info     ] Graph projection completed: 20 nodes, 32 edges in 0.00s [CogneeGraph]

2026-03-13T07:52:45.710714 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 20, 'connection_count': 32}

Search results: [{'dataset_id': UUID('e51766c1-b67c-53ee-9f9a-8cc9bac0c2b2'), 'dataset_name': 'main_dataset', 'dataset_tenant_id': None, 'search_result': ['- Globex Analytics: Three-year contract (May 2024–May 2027), £2.3M.\n- Acme Ro

Here are the details of the contracts:

1. **Globex Analytics**: 
   - **Terms**: Three-year contract (May 2024–May 2027)
   - **Value**: £2.3M

2. **Acme Rocket Skates Ltd**: 
   - **Terms**: Five-year contract (Oct 2024–Oct 2029)
   - **Value**: £4.2M


In [8]:
from cognee_integration_langgraph import get_sessionized_cognee_tools

add_tool, search_tool = get_sessionized_cognee_tools()  # session-aware tools
fresh_agent = create_react_agent("openai:gpt-4o-mini", tools=[add_tool, search_tool])
fresh_agent.step_timeout = None

response = await fresh_agent.ainvoke(
    {"messages": [HumanMessage(content="What contracts did we sign and what are their values?")]}
)
print(response["messages"][-1].content)


Initialized session with session_id = cognee-test-user-aa7b721b-159f-44c7-aab8-f8ea2d3519b7
/var/folders/85/g_scmr8n4w1bk9_6vq_klj5r0000gn/T/ipykernel_66290/2152611719.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  fresh_agent = create_react_agent("openai:gpt-4o-mini", tools=[add_tool, search_tool])

Using tool search_tool with session_id: cognee-test-user-aa7b721b-159f-44c7-aab8-f8ea2d3519b7

Searching cognee for: contracts and their values with session_id: cognee-test-user-aa7b721b-159f-44c7-aab8-f8ea2d3519b7, node_set: None, query_type: None

2026-03-13T07:53:16.364954 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.28s [cognee.shared.logging_utils]

2026-03-13T07:53:16.365714 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-03-13T07:53:16

We signed the following contracts:

1. **Globex Analytics**: £2.3M
2. **Acme Rocket Skates Ltd**: £4.2M
